In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

# GridWorld and LunarLander Plots for PDA

During the original submission of PDA, we did not re-run GridWorld and LunarLander since 1) I did not want to run more experiments and 2) at first glance, I did not due any tuning originally (a reviewer asked us to re-run experiments we did tune for a more fair comparison). 

A couple weeks later, George asked for GridWorld plots but in a new style. Since I did not have the data, I re-ran it. Here it is.

You can go to `2026_02_25.ipynb` for tuning.

### GridWorld-v0

In [ ]:
path = "/Users/calebju/Code/RL-general-action-state/logs"

# 02_25_2026/exp_0.py
n_seeds = 10
n_iters = 100_000
data_1 = np.zeros((5,n_seeds,n_iters))
aux_1 = np.zeros((5,n_seeds,n_iters), dtype=float)
len_1 = np.zeros((5, n_seeds), dtype=int)
for i in range(data_1.shape[0]):
    for j in range(n_seeds):
        df = pd.read_csv(os.path.join(path, "02_25_2026/exp_0/run_%d/seed=%d.csv" % (i,j)))
        temp = df['episode rewards']
        temp_ = df['episode len']
        len_1[i,j] = len(temp)
        data_1[i,j,:len_1[i,j]] = temp
        aux_1[i,j,:len_1[i,j]] = np.cumsum(temp_)

Clean it.

In [ ]:
xs = np.append(
    np.append(np.arange(1_000, step=10), np.arange(1_000, 10_000, step=25)), 
    np.arange(10_000, 95_000, step=100)
)
clean_1 = np.zeros((data_1.shape[0], n_seeds, len(xs)), dtype=int)
for i in range(clean_1.shape[0]):
    for j in range(n_seeds):
        for k,x in enumerate(xs):
            # finds index i such that len_i < x <= len_(i+1)
            clean_1[i,j,k] = np.argmax(x <= aux_1[i,j,:])
            if clean_1[i,j,k] < clean_1[i,j,k-1]:
                print("Wrong index at (i,j,k)=(%d,%d,%d) | x=%d max=%d" % (i,j,k,x,np.max(aux_1[i,j,:])))

Now we plot the performance.

In [ ]:
plt.style.use('ggplot')
_, ax = plt.subplots(figsize=(5,4))
label_arr = ['pda (linear-kl)', 'pda (nn-kl)', 'pda (nn-tsallis)', 'ppo', 'dqn']
color_arr = ['red', 'green', 'purple', 'blue', 'black']
lss_arr = ["solid","dashed","dotted","dashdot",(1,(3,5,1))]
ones_5 = 0.2*np.ones(5)
offs = len(ones_5)-1

aa = [0,1,2,3,4]
for i in aa:
    ys = np.zeros(clean_1.shape[1:])
    for j in range(n_seeds):
        ys[j] = data_1[i, j, clean_1[i,j]]
        ys[j] = np.convolve(ys[j], ones_5)[:-len(ones_5)+1]
    med = np.mean(ys, axis=0)
    rng = np.std(ys, axis=0)
    rng *= 2.571 # based on 2-sided t-score with p=0.05
    ax.plot(xs[offs:], -med[offs:], label=label_arr[i], color=color_arr[i], linestyle=lss_arr[i])
    ax.fill_between(xs[offs:], (-med-rng)[offs:], (-med+rng)[offs:], color=color_arr[i], alpha=0.1)

ax.legend(loc="upper left")
ax.set(
    title="Costs in GridWorld over 10 trials",
    ylabel="Cumulative discounted cost\n smoothed over %d periods" % len(ones_5),
    xlabel="Samples",
    ylim=(-50,350),
    xlim=(-500, 95_000),
)

plt.tight_layout()
# plt.savefig("humanoid.png", dpi=90)
plt.savefig("gridworld_reran.png", dpi=240)

Now we do seed-to-seed.

In [ ]:
plt.style.use('ggplot')
_, axes = plt.subplots(ncols=2, figsize=(7,4))
ones_5 = 0.2*np.ones(5)
offs = len(ones_5)-1

aa = [1,3]
for k,i in enumerate(aa):
    for j in range(n_seeds):
        ys = np.convolve(data_1[i, j, clean_1[i,j]], ones_5)[:-len(ones_5)+1]
        axes[k].plot(xs[offs:], -ys[offs:])

axes[0].set(
    title="pda on GridWorld",
    ylabel="Cumulative discounted cost\n smoothed over %d periods" % len(ones_5),
    xlabel="Samples",
    ylim=(0,250),
    xlim=(-500, 95_000),
)
axes[1].set(
    title="ppo on GridWorld",
    xlabel="Samples",
    ylim=(0,250),
    xlim=(-500, 95_000),
)

plt.tight_layout()
# plt.savefig("humanoid.png", dpi=90)
plt.savefig("gridworld_seed2seed_reran.png", dpi=240)

### LunarLander-v3

In [ ]:
path = "/Users/calebju/Code/RL-general-action-state/logs"
n_seeds = 5

# 02_12_2026/exp_0.py 
data_2 = np.zeros((5,n_seeds,500_000), dtype=float)
aux_2 = np.zeros((5,n_seeds,500_000), dtype=float)
len_2 = np.zeros((5, n_seeds), dtype=int)
for i in range(len(data_2)):
    for j in range(n_seeds):
        # df = pd.read_csv(os.path.join(path, "02_12_2026/exp_1/run_%d/seed=%d.csv" % (i, j)))
        df = pd.read_csv(os.path.join(path, "02_25_2026/exp_1/run_%d/seed=%d.csv" % (i,j)))
        temp = df['episode rewards']
        temp2 = df['episode len']
        len_2[i,j] = len(temp)
        # data_1[i,j,:len_1[i,j]] = np.convolve(temp, ones_5)[:-2]
        data_2[i,j,:len_2[i,j]] = temp
        aux_2[i,j,:len_2[i,j]] = np.cumsum(temp2)

Bucket the data.

In [ ]:
xs = np.append(
    np.append(np.arange(1_000, step=10), np.arange(1_000, 10_000, step=25)), 
    np.arange(10_000, 99_250, step=100)
)
clean_2 = np.zeros((data_2.shape[0], n_seeds, len(xs)), dtype=int)
for i in range(clean_2.shape[0]):
    for j in range(clean_2.shape[1]):
        for k,x in enumerate(xs):
            # finds index i such that len_i < x <= len_(i+1)
            clean_2[i,j,k] = np.argmax(x <= aux_2[i,j,:])
            # finds index i such that len_i < x <= len_(i+1)
            if k > 0:
                if clean_2[i,j,k] < clean_2[i,j,k-1]:
                    print("Wrong index at (i,j,k)=(%d,%d,%d) | x=%d max=%d" % (i,j,k,x,np.max(aux_2[i,j,:])))

Plot.

In [ ]:
plt.style.use('ggplot')
_, ax = plt.subplots(figsize=(5,4))
label_arr = ['pda (linear-kl)', 'pda (nn-kl)', 'pda (nn-tsallis)', 'ppo', 'dqn']
ones_5 = 0.05*np.ones(20)

aa = [0,1,2,3,4]
for i in aa:
    ys = np.zeros(clean_2.shape[1:])
    for j in range(n_seeds):
        ys[j] = data_2[i, j, clean_2[i,j]]
        ys[j] = np.convolve(ys[j], ones_5)[:-len(ones_5)+1]
    med = np.mean(ys, axis=0)
    rng = np.std(ys, axis=0)
    rng *= 2.571 # based on 2-sided t-score with p=0.05
    ax.plot(xs, -med, label=label_arr[i], color=color_arr[i], linestyle=lss_arr[i])
    ax.fill_between(xs, -med-rng, -med+rng, color=color_arr[i], alpha=0.1)

ax.legend(loc="upper right")
ax.set(
    title="Costs in LunarLander over 10 trials",
    ylabel="Cumulative discounted cost\n smoothed over %d periods" % len(ones_5),
    xlabel="Samples",
    ylim=(-75,300),
    xlim=(-2_000, 99_250),
)

plt.tight_layout()
# plt.savefig("humanoid.png", dpi=90)
plt.savefig("lunarlander_reran.png", dpi=240)